## Connexion à PostgreSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# ENV SETUP

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

encoded_password = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{encoded_password}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

## Création des tables

In [ ]:
from sqlalchemy import create_engine, text

sql = """
-- Table des agences
CREATE TABLE IF NOT EXISTS agences (
    agence_id SERIAL PRIMARY KEY,
    agence VARCHAR(100) UNIQUE
);

-- Table des segments clients
CREATE TABLE IF NOT EXISTS segments_client (
    segment_id SERIAL PRIMARY KEY,
    segment_client VARCHAR(50) UNIQUE,
    Categorie_risque VARCHAR(50)
);

-- Table des clients
CREATE TABLE IF NOT EXISTS clients (
    client_id VARCHAR PRIMARY KEY,
    segment_client VARCHAR(50),
    score_credit_client NUMERIC(10,2)
);

-- Table des produits
CREATE TABLE IF NOT EXISTS produits (
    produit_id SERIAL PRIMARY KEY,
    produit VARCHAR(100) UNIQUE
);

-- Table des transactions
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id BIGINT PRIMARY KEY,
    client_id VARCHAR,
    date_transaction DATE,
    montant NUMERIC(15,2),
    devise VARCHAR(10),
    taux_change_eur NUMERIC(10,4),
    montant_eur NUMERIC(15,2),
    categorie VARCHAR(100),
    produit VARCHAR(100),
    agence VARCHAR(100),
    type_operation VARCHAR(50),
    statut VARCHAR(50),
    score_credit_client NUMERIC(10,2),
    segment_client VARCHAR(50),
    solde_avant NUMERIC(15,2),
    anomaly_montant NUMERIC(10,2),
    anomaly_score NUMERIC(10,2),
    is_anomaly BOOLEAN,
    "Années" INT,
    "Mois" INT,
    "Trimestre" INT,
    marge NUMERIC(10,2),
    "jour-semaine" VARCHAR(20),
    montant_eur_verifie NUMERIC(15,2),
    Comparaison VARCHAR(50),
    Categorie_risque VARCHAR(50),
    taux_rejet NUMERIC(10,2),
    CONSTRAINT fk_transactions_clients
        FOREIGN KEY (client_id) REFERENCES clients(client_id)
);
"""

with engine.begin() as connection:
    connection.execute(text(sql))

print("✅ Tables créées avec succès")

In [ ]:
import pandas as pd

df = pd.read_csv('financecore_clean (3).csv')
df

In [ ]:
df.columns

In [ ]:
agences_df = df["agence"].drop_duplicates()
segments_clients_df = df[["segment_client", "categorie_risque"]].drop_duplicates()
clients_df = df[["client_id", "segment_client", "score_credit_client"]].drop_duplicates()
produits_df = df[["produit"]].drop_duplicates()
transactions_df = df[[
        "transaction_id",
        "client_id",
        "date_transaction",
        "montant",
        "devise",
        "taux_change_eur",
        "montant_eur",
        "categorie",
        "produit",
        "agence",
        "type_operation",
        "statut",
        "score_credit_client",
        "segment_client",
        "solde_avant",
        "z_score_montant",
        "outlier_zscore",
        "is_anomaly",
        "annee",
        "mois",
        "trimestre",
        "jour_semaine",
        "montant_eur_verifie",
        "categorie_risque",
        "taux_rejet"
    ]].drop_duplicates(subset="transaction_id")

In [ ]:
df.columns

In [ ]:
agences_df.to_sql("agences", engine, if_exists="replace", index=False)
segments_clients_df.to_sql("segments_client", engine, if_exists="replace", index=False)
produits_df.to_sql("produits", engine, if_exists="replace", index=False)
transactions_df.to_sql("transactions", engine, if_exists="replace", index=False)
clients_df.to_sql("clients", engine, if_exists="replace", index=False)

print("✅ Données insérées avec succès dans toutes les tables")

In [ ]:
df.columns

In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            agence,
            produit,
            "annee" AS annee,
            "mois" AS mois,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS total_montant_eur,
            AVG(montant_eur) AS moyenne_montant_eur
        FROM
            transactions
        GROUP BY
            agence,
            produit,
            "annee",
            "mois"
        HAVING
            SUM(montant_eur) > 1000
        ORDER BY
            total_montant_eur DESC
        LIMIT 10;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
        SELECT
            client_id,
            solde_avant
        FROM
            transactions
        WHERE
            solde_avant < (
                SELECT AVG(solde_avant)
                FROM transactions
            )
        ORDER BY
            solde_avant
        LIMIT 10;
'''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
SELECT
    "categorie_risque",
    COUNT(*) AS nombre_total_transactions,
    SUM(
        CASE 
            WHEN statut IN ('Rejeté', 'Refusé', 'Défaut') THEN 1
            ELSE 0
        END
    ) AS nombre_defauts,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN statut IN ('Rejeté', 'Refusé', 'Défaut') THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS taux_defaut_pourcentage
FROM
    transactions
GROUP BY
    "categorie_risque"
ORDER BY
    taux_defaut_pourcentage DESC;
'''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
SELECT
    t.transaction_id,
    t.date_transaction,
    t.montant_eur,
    c.client_id,
    c.segment_client,
    sc."categorie_risque",
    a.agence,
    p.produit,
    t.statut
FROM
    transactions t
INNER JOIN
    clients c ON t.client_id = c.client_id
INNER JOIN
    segments_client sc ON c.segment_client = sc.segment_client
INNER JOIN
    agences a ON t.agence = a.agence
INNER JOIN
    produits p ON t.produit = p.produit
ORDER BY
    t.date_transaction DESC
    LIMIT 10;
'''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_transactions AS
SELECT
    t.agence,
    t.produit,
    t."annee" AS annee,
    t."mois" AS mois,
    COUNT(*) AS nombre_transactions,
    SUM(t.montant_eur) AS montant_total_eur,
    AVG(t.montant_eur) AS montant_moyen_eur
FROM
    transactions t
GROUP BY
    t.agence,
    t.produit,
    t."annee",
    t."mois";
'''))
        
with engine.begin() as conn:
        result = conn.execute(text('''
CREATE OR REPLACE VIEW vw_kpi_taux_defaut AS
SELECT
    sc."categorie_risque",
    COUNT(*) AS nombre_transactions,
    COUNT(*) FILTER (
        WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
    ) AS nombre_defauts,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS taux_defaut_pourcentage
FROM
    transactions t
JOIN
    clients c ON t.client_id = c.client_id
JOIN
    segments_client sc ON c.segment_client = sc.segment_client
GROUP BY
    sc."categorie_risque";
'''))
        
with engine.begin() as conn:
        result = conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_performance_agence AS
SELECT
    t.agence,
    COUNT(*) AS nombre_transactions,
    SUM(t.montant_eur) AS montant_total,
    AVG(t.montant_eur) AS montant_moyen,
    COUNT(DISTINCT t.client_id) AS nombre_clients_uniques
FROM
    transactions t
GROUP BY
    t.agence;
'''))